In [1]:
import xarray as xr
import numpy as np
import functions
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import seaborn as sns
import cmcrameri
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.markers import MarkerStyle
import statsmodels.api as sm
import pymannkendall as mk
from global_land_mask import globe

In [3]:
rpath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/'
timeslice_trend = slice(2003, 2025)
Arctic_lim = 60

### Land surface temperature

In [143]:
filepath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/temperature2m_ERA5-Land.nc'
temp_ERA5_Land = xr.open_dataset(filepath)
temp_ERA5_Land = temp_ERA5_Land.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
temp_ERA5_Land = temp_ERA5_Land.reindex(lat=list(reversed(temp_ERA5_Land.lat)))

In [13]:
filepath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/temperature2m_ERA5.nc'
temp_ERA5 = xr.open_dataset(filepath)
temp_ERA5 = temp_ERA5.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
temp_ERA5 = temp_ERA5.reindex(lat=list(reversed(temp_ERA5.lat)))

In [ ]:
season='JJA'
# Compute JJA trend over land for ERA5-Land
# Regional average, Arctic excluding Greenland (lons > 300)
temp_ERA5_Arctic_land = functions.computeWeightedMean(temp_ERA5_Land)
# Select season
temp_ERA5_season = temp_ERA5_Arctic_land.sel(time=temp_ERA5_Arctic_land.time.dt.season==season)
# Annual season mean
temp_ERA5_season_annual = temp_ERA5_season.groupby(temp_ERA5_season.time.dt.year).mean('time')
# Save file for later use
temp_ERA5_season_annual.to_netcdf(rpath+'temperature2m_ERA5_JJA_annual.nc')

In [114]:
# Regional average, Arctic excluding Greenland (lons > 300)
temp_ERA5_Arctic_land = functions.computeWeightedMean(temp_ERA5_Land)
# Annual mean
temp_ERA5_season_annual = temp_ERA5_Arctic_land.groupby(temp_ERA5_Arctic_land.time.dt.year).mean('time')
# Save file for later use
temp_ERA5_season_annual.to_netcdf(rpath+'temperature2m_ERA5_annual.nc')

#### Soil moisture ERA5-Land

In [4]:
# OPEN DATASET

filepath_1 = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/data_stream-moda.nc'
filepath_2 = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/SM_level2_ERA5-Land.nc'
SM_ERA5_1 = xr.open_dataset(filepath_1)
SM_ERA5_2 = xr.open_dataset(filepath_2)

SM_ERA5_1 = SM_ERA5_1.sel(valid_time=slice('1970-01-01','2025-09-01'))
SM_ERA5_1 = SM_ERA5_1.sel(valid_time=SM_ERA5_1.valid_time.dt.season=='JJA')
SM_ERA5 = xr.merge([SM_ERA5_1, SM_ERA5_2],compat='override')

In [5]:
# CREATE SOIL LEVEL FOR 20 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (20-7)/(28-7)
SM_ERA5['swvl_20cm'] = SM_ERA5['swvl1']+((20-7)/(28-7))*SM_ERA5['swvl2']

In [6]:
# CREATE SOIL LEVEL FOR 10 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (10-7)/(28-7)
SM_ERA5['swvl_10cm'] = SM_ERA5['swvl1']+weighting_fraction*SM_ERA5['swvl2']

In [7]:
# RE-INDEX

SM_ERA5 = SM_ERA5.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
SM_ERA5_1970_2025 = SM_ERA5.sel(time=slice('1970-01-01','2025-09-01'))
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.reindex(lat=list(reversed(SM_ERA5_1970_2025.lat)))
lons = np.array(SM_ERA5_1970_2025.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_ERA5_1970_2025.coords['lon'] = lons
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.sortby(SM_ERA5_1970_2025.lon)
SM_ERA5_1970_2025.to_netcdf(rpath+'SM_ERA5_1970_2025_10cm_20cm.nc')

In [57]:
filepath='/nird/datalake/NS9560K/diagnostics/ILAMB-Data/DATA/mrsos/WangMao/mrsos_olc.nc'

SM_WM = xr.open_dataset(filepath)
SM_WM_1970_2016 = SM_WM.sel(time=slice('1970-01-01','2016-12-31'),lat=slice(59,90))
SM_WM_1970_2016_masked = SM_WM_1970_2016.where(SM_WM_1970_2016['mrsos'] < 1000)
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.drop_vars('time_bounds')
lons = np.array(SM_WM_1970_2016_masked.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_WM_1970_2016_masked.coords['lon'] = lons
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.sortby(SM_WM_1970_2016_masked.lon)
SM_WM_1970_2016_masked.to_netcdf(rpath+'SM_WangMao_1970_2016_masked.nc')

In [ ]:
SM_ERA5_trends_2002_2024 = xr.open_dataset(rpath+'soil_moisture_ERA5/SM_ERA5_trends_JJA_2002_2024.nc')
SM_ERA5_trends_2002_2024 = SM_ERA5_trends_2002_2024.rename({'latitude':'lat', 'longitude':'lon'})
SM_ERA5_trends_2002_2024 = SM_ERA5_trends_2002_2024.reindex(lat=list(reversed(SM_ERA5_trends_2002_2024.lat)))
lons = np.array(SM_ERA5_trends_2002_2024.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_ERA5_trends_2002_2024.coords['lon'] = lons
SM_ERA5_trends_2002_2024 = SM_ERA5_trends_2002_2024.sortby(SM_ERA5_trends_2002_2024.lon)
SM_ERA5_trends_2002_2024.to_netcdf(rpath+'soil_moisture_ERA5/SM_ERA5_trends_JJA_2002_2024.nc')

In [7]:
SM_ERA5 = xr.open_dataset(rpath+'SM_ERA5_1970_2025_10cm_20cm.nc')

In [4]:
SM_WM = xr.open_dataset(rpath+'SM_WangMao_1970_2016_masked.nc')

In [ ]:
CERES = xr.open_dataset(rpath+'CERES_EBAF-TOA_Ed4.2.1_Subset_200003-202512.nc')
CERES = CERES.sel(time=slice('2003-01-01','2025-12-1'))

for season in ["JJA"]:
    ds_season = CERES['cldarea_total_daynight_mon'].sel(time=CERES.time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')

    trends = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    p_values = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    for ilon in range(len(CERES.lon)):
        for ilat in range(len(CERES.lat)):
            y = ds_annual.isel(lon=ilon, lat=ilat).values
            test = mk.original_test(y)

            trends[ilon, ilat] = test.slope
            p_values[ilon, ilat] = test.p
    
    ds_annual['trend'] = ({'lon':CERES.lon, 'lat':CERES.lat}, trends)
    ds_annual['p_value'] = ({'lon':CERES.lon, 'lat':CERES.lat}, p_values)

    ds_annual.to_netcdf(rpath+"CERES_trends_"+season+"_2003_2025.nc")

AttributeError: 'numpy.ndarray' object has no attribute 'slope'

In [ ]:
for season in ["JJA"]:
    ds_season = CERES['cldarea_total_daynight_mon'].sel(time=CERES.time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')
    
    x = ds_annual.year.values
    weights = np.ones(len(x))
    # Add a smaller weight to first point, as it does not include June
    weights[0] = 2/3
    X = sm.add_constant(x)

    trends = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    p_values = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    for ilon in range(len(CERES.lon)):
        for ilat in range(len(CERES.lat)):
            y = ds_annual.isel(lon=ilon, lat=ilat).values
            model = sm.WLS(y, X, weights=weights)
            CERES_fit = model.fit()

            trends[ilon, ilat] = CERES_fit.params[1]
            p_values[ilon, ilat] = CERES_fit.pvalues[1]
    
    ds_annual['trend'] = ({'lon':CERES.lon, 'lat':CERES.lat}, trends)
    ds_annual['p_value'] = ({'lon':CERES.lon, 'lat':CERES.lat}, p_values)

    ds_annual.to_netcdf(rpath+"CERES_trends_"+season+"_2002_2025.nc")

In [ ]:
#CERES = xr.open_dataset(rpath+'CERES_EBAF-TOA_Ed4.2_Subset_200207-202407.nc')
CERES = xr.open_dataset(rpath+'CERES_EBAF-TOA_Ed4.2.1_Subset_200003-202512.nc')
CERES = CERES.sel(time=slice('2003-01-01','2025-12-1'))

# Select season
timeslice_trend=slice(2003,2025)
CERES_season = CERES_Arctic.sel(time=CERES_Arctic.time.dt.season==season)
# Annual season mean
CERES_season_annual = CERES_season.groupby(CERES_season.time.dt.year).mean('time')
# Create weighted linear regression
x = CERES_season_annual.year.values
x = x
y = CERES_season_annual.values
weights = np.ones(len(x))

# Fit weighted least squares regression model
X = sm.add_constant(x)
model = sm.WLS(y, X, weights=weights)
CERES_fit = model.fit()
CERES_slope = CERES_fit.params[1]
CERES_intercept = CERES_fit.params[0]
CERES_p_value = CERES_fit.pvalues[1]
CERES_std_err = CERES_fit.bse[1]
print(CERES_fit.summary())

for season in ["JJA"]:
    ds_season = CERES['cldarea_total_daynight_mon'].sel(time=CERES.time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')
    
    x = ds_annual.year.values
    weights = np.ones(len(x))
    # Add a smaller weight to first point, as it does not include June
    weights[0] = 2/3
    X = sm.add_constant(x)

    trends = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    p_values = np.ones((len(CERES.lon),len(CERES.lat)))*np.nan
    for ilon in range(len(CERES.lon)):
        for ilat in range(len(CERES.lat)):
            y = ds_annual.isel(lon=ilon, lat=ilat).values
            model = sm.WLS(y, X, weights=weights)
            CERES_fit = model.fit()

            trends[ilon, ilat] = CERES_fit.params[1]
            p_values[ilon, ilat] = CERES_fit.pvalues[1]
    
    ds_annual['trend'] = ({'lon':CERES.lon, 'lat':CERES.lat}, trends)
    ds_annual['p_value'] = ({'lon':CERES.lon, 'lat':CERES.lat}, p_values)

    ds_annual.to_netcdf(rpath+"CERES_trends_"+season+"_2002_2025.nc")

                            WLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.173
Model:                            WLS   Adj. R-squared:                  0.136
Method:                 Least Squares   F-statistic:                     4.617
Date:                Wed, 18 Mar 2026   Prob (F-statistic):             0.0429
Time:                        17:39:46   Log-Likelihood:                -40.430
No. Observations:                  24   AIC:                             84.86
Df Residuals:                      22   BIC:                             87.22
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        244.0488     80.884      3.017      0.0

In [6]:
rpath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/'
CERES = xr.open_dataset(rpath+'CERES_EBAF-TOA_Ed4.2_Subset_200207-202407.nc')

datadir = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/other_cloud_vars/'
landfrac = xr.open_dataarray(datadir+'LANDFRAC_piClim.nc')
landmask = landfrac.isel(time=0)
oceanmask = 1-landmask

# Interpolate to 
landmask_CERES = landmask.interp_like(CERES.isel(time=0))
landmask_CERES = landmask_CERES.where(landmask_CERES>=0)
landmask_CERES = landmask_CERES.fillna(0)

oceanmask_CERES = 1-landmask_CERES

In [65]:
SM_ERA5

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 168, lat: 301, lon: 3600)
Coordinates:
  * time       (time) datetime64[ns] 1kB 1970-06-01 1970-07-01 ... 2025-08-01
    expver     (time) <U4 3kB ...
  * lat        (lat) float64 2kB 60.0 60.1 60.2 60.3 ... 89.7 89.8 89.9 90.0
  * lon        (lon) float64 29kB 0.1 0.2 0.3 0.4 ... 359.7 359.8 359.9 360.0
    number     int64 8B ...
Data variables:
    swvl1      (time, lat, lon) float32 728MB ...
    swvl2      (time, lat, lon) float32 728MB nan nan nan nan ... nan nan nan
    swvl_20cm  (time, lat, lon) float32 728MB ...
    swvl_10cm  (time, lat, lon) float32 728MB nan nan nan nan ... nan nan nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-30T14:21 GRIB to CDM+CF via cfgrib-0.9.1...

In [66]:
SM_ERA5

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 168, lat: 301, lon: 3600)
Coordinates:
  * time       (time) datetime64[ns] 1kB 1970-06-01 1970-07-01 ... 2025-08-01
    expver     (time) <U4 3kB ...
  * lat        (lat) float64 2kB 60.0 60.1 60.2 60.3 ... 89.7 89.8 89.9 90.0
  * lon        (lon) float64 29kB 0.1 0.2 0.3 0.4 ... 359.7 359.8 359.9 360.0
    number     int64 8B ...
Data variables:
    swvl1      (time, lat, lon) float32 728MB ...
    swvl2      (time, lat, lon) float32 728MB nan nan nan nan ... nan nan nan
    swvl_20cm  (time, lat, lon) float32 728MB ...
    swvl_10cm  (time, lat, lon) float32 728MB nan nan nan nan ... nan nan nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-30T14:21 GRIB to CDM+CF via cfgrib-0.9.1...

In [72]:
# Select season
timeslice_trend=slice('2002-01-01','2025-12-01')
SM_ERA5_slice = SM_ERA5.sel(time=timeslice_trend)

# Annual season mean
ds_season = SM_ERA5_slice['swvl_10cm'].sel(time=SM_ERA5_slice.time.dt.season=='JJA')
ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')

x = ds_annual.year.values
X = sm.add_constant(x)

trends = np.ones((len(SM_ERA5_slice.lon),len(SM_ERA5_slice.lat)))*np.nan
p_values = np.ones((len(SM_ERA5_slice.lon),len(SM_ERA5_slice.lat)))*np.nan
for ilon in range(len(SM_ERA5_slice.lon)):
    for ilat in range(len(SM_ERA5_slice.lat)):
        y = ds_annual.isel(lon=ilon, lat=ilat).values*100
        model = sm.WLS(y, X)
        fit = model.fit()

        trends[ilon, ilat] = fit.params[1]
        p_values[ilon, ilat] = fit.pvalues[1]

ds_annual['trend'] = ({'lon':SM_ERA5_slice.lon, 'lat':SM_ERA5_slice.lat}, trends)
ds_annual['p_value'] = ({'lon':SM_ERA5_slice.lon, 'lat':SM_ERA5_slice.lat}, p_values)

ds_annual.to_netcdf(rpath+"SM_ERA5_trends_"+season+"_2002_2025.nc")

In [ ]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = SM_ERA5.sel(valid_time=SM_ERA5.valid_time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.valid_time.dt.year).mean('valid_time')
    converted_ds_annual = ds_annual['swvl_10cm']*100 # from m3/m3 to kg/m2

    trends = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    p_values = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    for ilon in range(len(SM_ERA5.longitude)):
        for ilat in range(len(SM_ERA5.latitude)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(converted_ds_annual.year, converted_ds_annual.isel(longitude=ilon, latitude=ilat))
            trends[ilon, ilat] = slope
            p_values[ilon, ilat] = p_value
    
    ds_annual['trend'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, trends)
    ds_annual['p_value'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, p_values)

    ds_annual.to_netcdf(savedir+"SM_ERA5_trends_"+season+"_2002_2025.nc")

In [70]:
var = 'swvl1'
ds_season = SM_ERA5.sel(valid_time=SM_ERA5.valid_time.dt.season==season)
ds_annual = ds_season.groupby(ds_season.valid_time.dt.year).mean('valid_time')
converted_ds_annual = ds_annual[var]*70 # from m3/m3 to kg/m2

In [ ]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = SM_ERA5.sel(valid_time=SM_ERA5.valid_time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.valid_time.dt.year).mean('valid_time')
    converted_ds_annual = ds_annual[var]*70 # from m3/m3 to kg/m2

    trends = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    p_values = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    for ilon in range(len(SM_ERA5.longitude)):
        for ilat in range(len(SM_ERA5.latitude)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(converted_ds_annual.year, converted_ds_annual.isel(longitude=ilon, latitude=ilat))
            trends[ilon, ilat] = slope
            p_values[ilon, ilat] = p_value
    
    ds_annual['trend'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, trends)
    ds_annual['p_value'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, p_values)

    ds_annual.to_netcdf(savedir+"SM_ERA5_20cm_trends_"+season+".nc")


In [75]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = SM_ERA5.sel(valid_time=SM_ERA5.valid_time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.valid_time.dt.year).mean('valid_time')
    converted_ds_annual = ds_annual[var]*70 # from m3/m3 to kg/m2

    trends = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    p_values = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    for ilon in range(len(SM_ERA5.longitude)):
        for ilat in range(len(SM_ERA5.latitude)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(converted_ds_annual.year.sel(year=slice(2002,2024)), converted_ds_annual.isel(longitude=ilon, latitude=ilat).sel(year=slice(2002,2024)))
            trends[ilon, ilat] = slope
            p_values[ilon, ilat] = p_value
    
    ds_annual['trend'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, trends)
    ds_annual['p_value'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, p_values)

    ds_annual.to_netcdf(savedir+"SM_ERA5_trends_"+season+"_2002_2024.nc")


In [25]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"
var='swvl1'

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = SM_ERA5.sel(valid_time=SM_ERA5.valid_time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.valid_time.dt.year).mean('valid_time')
    converted_ds_annual = ds_annual[var]*70 # from m3/m3 to kg/m2

    trends = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    p_values = np.ones((len(SM_ERA5.longitude),len(SM_ERA5.latitude)))*np.nan
    for ilon in range(len(SM_ERA5.longitude)):
        for ilat in range(len(SM_ERA5.latitude)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(converted_ds_annual.year.sel(year=slice(1970,2016)), converted_ds_annual.isel(longitude=ilon, latitude=ilat).sel(year=slice(1970,2016)))
            trends[ilon, ilat] = slope
            p_values[ilon, ilat] = p_value
    
    ds_annual['trend'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, trends)
    ds_annual['p_value'] = ({'longitude':SM_ERA5.longitude, 'latitude':SM_ERA5.latitude}, p_values)

    ds_annual.to_netcdf(savedir+"SM_ERA5_trends_"+season+"_1970_2016.nc")


In [4]:
season="JJA"
rpath = "/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"
ds_annual = xr.open_dataset(rpath+"SM_ERA5_trends_"+season+".nc")
ds_annual = ds_annual.reindex(latitude=list(reversed(ds_annual.latitude)))

In [36]:
filepath='/nird/datalake/NS9560K/diagnostics/ILAMB-Data/DATA/mrsos/WangMao/mrsos_olc.nc'
ds_SM_WM = xr.open_dataset(filepath)

In [37]:
ds_SM_WM

<xarray.Dataset> Size: 1GB
Dimensions:      (time: 564, nb: 2, lat: 360, lon: 720)
Coordinates:
  * time         (time) object 5kB 1970-01-16 12:00:00 ... 2016-12-16 12:00:00
  * lat          (lat) float64 3kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon          (lon) float64 6kB -179.8 -179.2 -178.8 ... 178.8 179.2 179.8
Dimensions without coordinates: nb
Data variables:
    time_bounds  (time, nb) object 9kB ...
    mrsos        (time, lat, lon) float64 1GB ...
Attributes:
    title:         Observation-based global multilayer soil moisture products...
    version:       1
    institutions:  Oak Ridge National Laboratory
    source:        Offline land surface models, reanalysis, and satellite soi...
    history:       \n2021-09-14: downloaded https://drive.google.com/file/d/1...
    references:    \n@ARTICLE{Wang2021,\n  author = {Wang, Y. and Mao, J. and...
    comments:      \ntime_period: 1970-01 through 2016-12; temporal_resolutio...
    convention:    CF-1.8

In [11]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = ds_SM_WM.sel(time=ds_SM_WM.time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')

    trends = np.ones((len(ds_SM_WM.lat),len(ds_SM_WM.lon)))*np.nan
    p_values = np.ones((len(ds_SM_WM.lat),len(ds_SM_WM.lon)))*np.nan
    for ilat in range(len(ds_SM_WM.lat)):
        for ilon in range(len(ds_SM_WM.lon)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(ds_annual.year, ds_annual['mrsos'].isel(lat=ilat, lon=ilon))
            trends[ilat, ilon] = slope
            p_values[ilat, ilon] = p_value
    
    ds_annual['trend'] = ({'lat':ds_SM_WM.lat, 'lon':ds_SM_WM.lon}, trends)
    ds_annual['p_value'] = ({'lat':ds_SM_WM.lat, 'lon':ds_SM_WM.lon}, p_values)

    ds_annual.to_netcdf(savedir+"SM_WM_trends_"+season+".nc")

In [19]:
from scipy import stats
savedir="/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/soil_moisture_ERA5/"

#for season in ["DJF", "MAM","JJA","SON"]:
for season in ["JJA"]:
    ds_season = ds_SM_WM.sel(time=ds_SM_WM.time.dt.season==season)
    ds_annual = ds_season.groupby(ds_season.time.dt.year).mean('time')

    trends = np.ones((len(ds_SM_WM.lat),len(ds_SM_WM.lon)))*np.nan
    p_values = np.ones((len(ds_SM_WM.lat),len(ds_SM_WM.lon)))*np.nan
    for ilat in range(len(ds_SM_WM.lat)):
        for ilon in range(len(ds_SM_WM.lon)):
            slope, intercept, r_value, p_value, std_err = stats.linregress(ds_annual.year.sel(year=slice(1994,2016)), ds_annual['mrsos'].isel(lat=ilat, lon=ilon).sel(year=slice(1994,2016)))
            trends[ilat, ilon] = slope
            p_values[ilat, ilon] = p_value
    
    ds_annual['trend'] = ({'lat':ds_SM_WM.lat, 'lon':ds_SM_WM.lon}, trends)
    ds_annual['p_value'] = ({'lat':ds_SM_WM.lat, 'lon':ds_SM_WM.lon}, p_values)

    ds_annual.to_netcdf(savedir+"SM_WM_trends_"+season+"_1994_2016.nc")